In [ ]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf

TICKERS_POOL = ["AAPL", "MSFT", "NVDA", "IBE.MC", "BBVA.MC", "CAF.MC", "GOOGL", "CIE.MC"]

def obtener_datos_financieros(
    n_activos: int = 4,
    fecha_inicio: str = "2025-01-01",
    fecha_fin: str = "2026-01-01",
    usar_cache: bool = True
):
    os.makedirs("data", exist_ok=True)
    tickers = TICKERS_POOL[:n_activos]
    cache_csv = f"data/precios_{n_activos}_activos.csv"
    output_json = f"data/parametros_{n_activos}_activos.json"

    precios_df = None

    if usar_cache and os.path.exists(cache_csv):
        precios_df = pd.read_csv(cache_csv, index_col=0, parse_dates=True)

    if precios_df is None or precios_df.empty:
        try:
            datos = yf.download(tickers, start=fecha_inicio, end=fecha_fin, progress=False)
            precios_df = datos["Close"] if "Close" in datos else datos["Adj Close"]
            precios_df = precios_df[tickers].dropna()
            precios_df.to_csv(cache_csv)
            print(f"[OK] Precios guardados en {cache_csv}")
        except Exception as e:
            print(f"[AVISO] Fallo de red ({e}). Generando datos sintéticos de respaldo...")
            np.random.seed(42)
            dias = 252
            retornos_sim = np.random.normal(0.0008, 0.018, size=(dias, n_activos))
            precios_sim = 100 * np.exp(np.cumsum(retornos_sim, axis=0))
            precios_df = pd.DataFrame(precios_sim, columns=tickers)
            precios_df.to_csv(cache_csv)

    # 3. Calcular retornos diarios porcentuales
    retornos_diarios = precios_df.pct_change().dropna()

    # 4. Anualizar vector de retornos (mu) y matriz de covarianzas (sigma)
    mu = (retornos_diarios.mean() * 252).to_numpy()
    sigma = (retornos_diarios.cov() * 252).to_numpy()

    # 5. Exportar el contrato de datos limpio(qaoa_model.py)
    payload = {
        "n_activos": n_activos,
        "tickers": tickers,
        "mu": mu.tolist(),
        "sigma": sigma.tolist()
    }
    with open(output_json, "w") as f:
        json.dump(payload, f, indent=4)

    print(f"[OK] Parámetros financieros listos en: {output_json}")
    return tickers, mu, sigma


if __name__ == "__main__":
    # Generamos de golpe los 3 escenarios del hackathon (4, 6 y 8 activos)
    for n in [8]:
        print(f"\n--- Procesando escenario de {n} activos ---")
        tickers, mu, sigma = obtener_datos_financieros(n_activos=n)
        print("Tickers:", tickers)
        print("Retornos esperados (mu):", np.round(mu, 4))
        print("Dimensión de covarianza (sigma):", sigma.shape)